##### Using coding agents for automating web activities
Exploration of building coding agents for web actions using selenium, helium and coding agents

In [1]:
from io import BytesIO
from time import sleep

import helium
from helpers import *
from PIL import Image
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from smolagents import CodeAgent, tool
from smolagents.agents import ActionStep

In [ ]:
# setup and configure the browser with chrome and configure screenshot capabilities
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--force-device-scale-factor=1")
chrome_options.add_argument("--window-size=1000,1350")
chrome_options.add_argument("--disable-pdf-viewer")
chrome_options.add_argument("--window-position=0,0")

# does not currently support non-headless mode
driver = helium.start_chrome(options=chrome_options, headless=True)

In [4]:
# setup a screenshot callback 
def save_screenshot(memory_step: ActionStep, agent: CodeAgent) -> None:
    sleep(1)
    driver = helium.get_driver()
    current_step = memory_step.step_number
    if driver is not None:
        for previous_memory_step in agent.memory.steps: # remove previous screenshots for lean processing
            if isinstance(previous_memory_step, ActionStep) and previous_memory_step.step_number <= current_step - 2:
                previous_memory_step.observations_images = None
            png_bytes = driver.get_screenshot_as_png()
            image = Image.open(BytesIO(png_bytes))
            print(f"captured a browser screenshot: { image.size } pixels")
            memory_step.observations_images = [image.copy()]
    # update observations with current ulr
    url_info = f"current url: {driver.current_url}"
    memory_step.observations = (
        url_info if memory_step.observations is None else memory_step.observations +  "\n" + url_info
    )

In [ ]:
# implement core tool for browser automation
@tool
def search_item_ctrl_f(text:str, nth_result: int = 1) -> str:
    """
    search for text on the current page via ctrl+f and jumps to the nth occurence
    args:
        text: the text to search for
        nth_result: which occurence to jump to (default is 1)
    returns:
        the text of the nth occurence
    """
    helium.write(text)
    helium.press(Keys.CONTROL + 'f')
    sleep(1)
    helium.press(Keys.DOWN)
    for _ in range(nth_result - 1):
        helium.press(Keys.DOWN)
    return helium.read()